In [45]:
import pandas as pd
import numpy as np

df = pd.read_csv("otodom_gdansk_rent_dirty.csv")
df_clean = df.dropna(subset=['price_pln']).copy()
df_clean["czynsz_pln"] = df_clean["czynsz_pln"].fillna(0)

In [46]:
def clean_floor(val):
  if pd.isna(val):
    return np.nan
  val = str(val).lower().strip()
  if "parter" in val:
    return 0
  if "poddasze" in val:
    return np.nan
  if '/' in val:
    return val.split('/')[0]
  return val

In [47]:
df_clean["floor_clean"] = df_clean["floor"].apply(clean_floor)
df_clean["floor_clean"] = pd.to_numeric(df_clean["floor_clean"], errors='coerce')

In [48]:
columns_to_keep = ["total_price_pln", "price_pln", "czynsz_pln", "area_m2", "price_per_m2", "rooms",
                   "floor_number", "year_built", "building_type",
                   "owner_type", "lat", "lon", "has_balcony", "has_terrace", "has_garden", "has_elevator", "has_garage",
                   "has_parking", "has_storage_room", "has_air_conditioning", "has_furnished", "has_dishwasher",
                   "has_washing_machine", "has_fridge", "has_oven", "has_intercom", "has_monitoring",
                   "has_protected_estate", "has_pets_allowed"]

In [49]:
df_clean["total_price_pln"] = df_clean["price_pln"] + df_clean["czynsz_pln"]
df_clean["price_per_m2"] = round((df_clean["total_price_pln"] / df_clean['area_m2']), 2)


In [50]:
mode_owner = df_clean['owner_type'].mode()[0]
df_clean['owner_type'] = df_clean['owner_type'].fillna(mode_owner)

In [51]:
median_floor = df_clean['floor_number'].median()
df_clean['floor_number'] = df_clean['floor_number'].fillna(median_floor)

In [52]:
df_clean['building_type'] = df_clean['building_type'].fillna('unknown')

df_clean['year_built_is_missing'] = df_clean['year_built'].isna()
df_clean['year_built'] = df_clean['year_built'].fillna(df_clean['year_built'].median())

In [53]:
df_clean['heating'] = df_clean['heating'].replace('brak informacji', 'unknown')
df_clean['heating'] = df_clean['heating'].fillna('unknown')

In [54]:
df_clean['building_material'] = df_clean['building_material'].fillna('unknown')

In [55]:
bool_cols = df_clean.select_dtypes(include='bool').columns

df_clean[bool_cols] = df_clean[bool_cols].astype(int)

In [56]:
df_clean = df_clean[df_clean['floor_number'] <= 20]

In [57]:
df_clean = df_clean[df_clean['total_price_pln'] >= 1000]

In [58]:
df_clean['czynsz_pln'] = df_clean['czynsz_pln'].abs()

In [59]:
ml_dataframe = df_clean[columns_to_keep]

In [60]:
ml_dataframe.to_csv('otodom_gdansk_rent_clean.csv', index=False)

In [61]:
import pandas as pd
import plotly.express as px

In [62]:
df = pd.read_csv('otodom_gdansk_rent_clean.csv')

In [63]:
fig = px.scatter_mapbox(
    df,
    lat="lat",
    lon="lon",
    color="total_price_pln",          # Color the dots by price
    size="area_m2",               # Make the dots bigger for larger apartment
    size_max = 10,
    color_continuous_scale=px.colors.sequential.Turbo,
    range_color=[2000, 6000],
    hover_name="total_price_pln",     # Show price when hovering over a dot
    hover_data=["rooms", "area_m2", "price_per_m2"],
    zoom=10,
    center={"lat": 54.3660, "lon": 18.5966}, # Centered on Gdańsk
    mapbox_style="carto-positron",
    title="Gdańsk Apartment Rental Prices"
)

In [64]:
fig.show()